In [ ]:
# libraries

from io import BytesIO

import boto3
import requests
from botocore.config import Config

In [ ]:
"""
FUNCTION
"""

def ingest_deprivation_data(
    bucket_name,
    aws_region,
    s3_prefix="raw/enrichment/deprivation"
):
    """
    Download the official English and Welsh deprivation
    datasets and upload the original CSV files to S3.

    English source
    --------------
    English Indices of Deprivation 2025, File 7.

    Welsh source
    ------------
    Welsh Index of Multiple Deprivation 2025 from the
    StatsWales API.

    Returns
    -------
    dict
        S3 locations for the English and Welsh CSV files.
    """

    # Create a web session
    web_session = requests.Session()

    # Create an S3 connection using locally configured credentials
    s3 = boto3.client(
        "s3",
        region_name=aws_region,
        config=Config(
            retries={
                "max_attempts": 10,
                "mode": "standard"
            }
        )
    )

    uploaded_files = {}

    # =========================================================
    # 1. ENGLISH INDICES OF DEPRIVATION 2025
    # =========================================================

    print("Downloading English deprivation data...")

    # Direct link to File 7 from the official GOV.UK release
    english_url = (
        "https://assets.publishing.service.gov.uk/"
        "media/691ded56d140bbbaa59a2a7d/"
        "File_7_IoD2025_All_Ranks_Scores_Deciles_"
        "Population_Denominators.csv"
    )

    english_response = web_session.get(
        english_url,
        timeout=180
    )

    english_response.raise_for_status()

    # Basic validation that some data was returned
    if not english_response.content:
        raise RuntimeError(
            "The English deprivation download was empty."
        )

    english_rows = (
        len(english_response.content.splitlines()) - 1
    )

    print(
        f"English CSV downloaded: "
        f"{len(english_response.content):,} bytes, "
        f"{english_rows:,} data rows."
    )

    # English S3 destination
    english_s3_key = (
        f"{s3_prefix}/england/"
        "File_7_IoD2025_All_Ranks_Scores_Deciles_"
        "Population_Denominators.csv"
    )

    print("Uploading English deprivation CSV to S3...")

    s3.upload_fileobj(
        Fileobj=BytesIO(english_response.content),
        Bucket=bucket_name,
        Key=english_s3_key,
        ExtraArgs={
            "ContentType": "text/csv"
        }
    )

    english_s3_location = (
        f"s3://{bucket_name}/{english_s3_key}"
    )

    # Confirm that the object exists
    english_object = s3.head_object(
        Bucket=bucket_name,
        Key=english_s3_key
    )

    print(f"Uploaded: {english_s3_location}")
    print(
        f"Uploaded size: "
        f"{english_object['ContentLength']:,} bytes."
    )

    uploaded_files["england"] = english_s3_location

    # =========================================================
    # 2. WELSH INDEX OF MULTIPLE DEPRIVATION 2025
    # =========================================================

    print("--------")
    print("Requesting Welsh deprivation data from StatsWales...")

    welsh_dataset_id = (
        "9706edd9-73ad-4902-bb12-7ccd7038626e"
    )

    welsh_api_base = (
        "https://api.stats.gov.wales/v2/"
        f"{welsh_dataset_id}"
    )

    # These options correspond to:
    # - current unfiltered table
    # - unformatted numbers
    # - include reference codes and hierarchies
    # - human-readable English column/value names
    welsh_download_options = {
        "filters": [],
        "options": {
            "use_raw_column_names": False,
            "use_reference_values": False,
            "data_value_type": "raw_extended"
        }
    }

    # Ask StatsWales to generate an ID for these options
    filter_response = web_session.post(
        f"{welsh_api_base}/data",
        json=welsh_download_options,
        timeout=60
    )

    filter_response.raise_for_status()

    filter_id = filter_response.json().get("filterId")

    if not filter_id:
        raise RuntimeError(
            "StatsWales did not return a download filter ID."
        )

    print(f"StatsWales filter created: {filter_id}")

    # Download the resulting table as CSV
    welsh_response = web_session.get(
        f"{welsh_api_base}/data/{filter_id}",
        params={
            "format": "csv"
        },
        timeout=180
    )

    welsh_response.raise_for_status()

    if not welsh_response.content:
        raise RuntimeError(
            "The Welsh deprivation download was empty."
        )

    welsh_rows = (
        len(welsh_response.content.splitlines()) - 1
    )

    print(
        f"Welsh CSV downloaded: "
        f"{len(welsh_response.content):,} bytes, "
        f"{welsh_rows:,} data rows."
    )

    # Welsh S3 destination
    welsh_s3_key = (
        f"{s3_prefix}/wales/"
        "wimd_2025_index_domain_ranks_groups_lsoa.csv"
    )

    print("Uploading Welsh deprivation CSV to S3...")

    s3.upload_fileobj(
        Fileobj=BytesIO(welsh_response.content),
        Bucket=bucket_name,
        Key=welsh_s3_key,
        ExtraArgs={
            "ContentType": "text/csv"
        }
    )

    welsh_s3_location = (
        f"s3://{bucket_name}/{welsh_s3_key}"
    )

    # Confirm that the object exists
    welsh_object = s3.head_object(
        Bucket=bucket_name,
        Key=welsh_s3_key
    )

    print(f"Uploaded: {welsh_s3_location}")
    print(
        f"Uploaded size: "
        f"{welsh_object['ContentLength']:,} bytes."
    )

    uploaded_files["wales"] = welsh_s3_location

    print("--------")
    print("Deprivation ingestion complete.")

    return uploaded_files

In [ ]:
"""
CALL
"""

deprivation_s3_locations = ingest_deprivation_data(
    bucket_name="rockborne-ch19-g1-crime",
    aws_region="us-west-2"
)